# Task 6 – Wu-Palmer + phonetic similarity

> *We would like to extend the Wu and Palmer semantic similarity extension of Lesk algorithm implemented in 3)
> by incorporating the phonetic similarity as well, so that the overall similarity is computed as a convex
> combination of Wu and Palmer similarity and phonetic similarity (after being normalized) with coefficient
> of 0.5. Suggest an implementation and run your new code on selected MSH-WSD dataset.*

In [39]:
import utils as u

import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
from nltk.corpus import wordnet as wn

from pywsd._wordnet import wn as pwn
from pywsd.similarity import sim
from pywsd.utils import lemmatize
from pywsd.tokenize import word_tokenize
import wn.compat.sensekey as sensekey

from pathlib import Path
import html, re, random, json, time
import pandas as pd

Our new similarity measure combines the Wu-Palmer similarity and the phonetic similarity. Wu-Palmer is already between 0 and 1, but the Levenshtein distance between two Soundex codes goes from 0 to 4 (and a higher value means *more different*). So we first normalize it with `1 - distance / 4`: now it is also between 0 and 1, and 1 means the two words sound the same.

In [40]:
def phonetic_similarity(w1, w2):
    """Phonetic similarity between two words, from 0 (completely different) to 1 (same sound).
    Soundex codes always have 4 characters, so the Levenshtein distance goes from 0 to 4."""
    distance = u.levenshtein(u.soundex(w1), u.soundex(w2))
    return 1 - distance / 4

# Test
for w1, w2 in [("drug", "drugs"), ("two", "to"), ("drug", "truck"), ("drug", "clinic")]:
    print(f"{w1:5} {u.soundex(w1)}  {w2:7} {u.soundex(w2)}  similarity = {phonetic_similarity(w1, w2):.2f}")

drug  D620  drugs   D620  similarity = 1.00
two   T000  to      T000  similarity = 1.00
drug  D620  truck   T620  similarity = 0.75
drug  D620  clinic  C452  similarity = 0.00


Wu-Palmer compares two **senses** (synsets), while our phonetic similarity compares two **words**. To combine them, also the phonetic part must compare two senses. A sense can have more than one name (lemma), e.g. `['drug', 'dose']`. So the phonetic similarity between two senses is the **highest** phonetic similarity between a lemma of the first sense and a lemma of the second one: it is enough that one of the names sounds similar.

In [41]:
def sense_phonetic(s1, s2):
    """Phonetic similarity between two senses: the best value among all the pairs of their lemmas."""
    return max(phonetic_similarity(a, b) for a in s1.lemmas() for b in s2.lemmas())

clinic = pwn.synsets("clinic")[0]
medicine = pwn.synsets("medicine")[0]
for s in pwn.synsets("drug"):
    print(f"{str(s.lemmas()):22} vs clinic {sense_phonetic(s, clinic):.2f}   vs medicine {sense_phonetic(s, medicine):.2f}")

['drug']               vs clinic 0.00   vs medicine 0.25
['drug', 'dose']       vs clinic 0.00   vs medicine 0.25
['drug', 'do drugs']   vs clinic 0.25   vs medicine 0.25


We can notice that the phonetic similarity between senses can give points to the wrong sense only by chance (e.g. *do drugs* vs *clinic*). 

Now we use it inside a WSD method, to see how it changes the chosen sense.
The method works like the Wu-Palmer method of pywsd, but with our new measure `alpha * WuPalmer + (1 - alpha) * phonetic`:
1. for each sense of the target word, we compare it with all the senses of each context word and keep only the **best** value;
2. we **sum** these best values over all the context words;
3. the sense with the **highest sum** wins.

With `alpha = 1` it is only Wu-Palmer, with `alpha = 0.5` it is the measure asked by the task. The target word is removed from the context, otherwise it would compare with itself.


In [42]:
def wup_phonetic(context, word, alpha=0.5):
    ctx = [lemmatize(w) for w in word_tokenize(context)]
    target = lemmatize(word)
    ctx = [w for w in ctx if w.lower() != target.lower()]
    scores = []
    for i in pwn.synsets(target):
        total = 0.0
        for j in ctx:
            best = 0.0
            for k in pwn.synsets(j):
                wup = sim(i, k, "wup") or 0.0
                best = max(best, alpha * wup + (1 - alpha) * sense_phonetic(i, k))
            total += best
        scores.append((total, i))
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores


sentence = "I have been prescribed two important drugs today during my visit to clinic"
for alpha in (1, 0.5):
    print(f"alpha = {alpha}")
    for total, s in wup_phonetic(sentence, "drug", alpha):
        print(f"   {total:5.2f}  {str(s.lemmas()):22} {s.definition()}")

alpha = 1
    2.61  ['drug']               a substance that is used as a medicine or narcotic
    1.20  ['drug', 'do drugs']   use recreational drugs
    1.09  ['drug', 'dose']       administer a drug to
alpha = 0.5
    2.95  ['drug', 'dose']       administer a drug to
    2.71  ['drug']               a substance that is used as a medicine or narcotic
    2.50  ['drug', 'do drugs']   use recreational drugs


Wu-Palmer alone is correct, but with the phonetic part the verb sense *administer a drug to* wins. This sense has two lemmas, *drug* and *dose*, and *dose* has a short Soundex code (D200) similar to the codes of short context words (*two*, *to*, *today* → T000, T300). So it gets a small bonus from many context words, only because of the sound.

Now we evaluate the new similarity on the same 39 words of Task 4, with the same sample of abstracts (10 per sense, same seed) and the same context window (10 words on each side). To compare, we try different weights for Wu-Palmer: `alpha = 1` (only Wu-Palmer), `alpha = 0.8` and `alpha = 0.5` (the measure asked by the task).

In [43]:
DATA_DIR = Path(r"C:\Users\flona\Documents\nlp_project\nlp_project_26\data\MSHCorpus")

concept_map = {}

with open(DATA_DIR / "benchmark_mesh.txt", encoding="iso-8859-1") as f:
    for line in f:
        parts = line.strip().split("\t")   
        word = parts[0]                   
        cuis = parts[1:]               
        concept_map[word] = {f"M{i}": cui for i, cui in enumerate(cuis, start=1)}

print("Number of ambiguous words:", len(concept_map))
print("AA ->", concept_map["AA"])
print("Ca ->", concept_map["Ca"])

def is_acronym(word):
    """An acronym has at least 2 capital letters or a digit, and no spaces."""
    if " " in word:
        return False
    return sum(c.isupper() for c in word) >= 2 or any(c.isdigit() for c in word)

MANUAL_ACRONYMS = {"Ca", "Ala", "Epi", "Fe", "dC", "pI", "Staph", "Strep"}
MANUAL_TERMS    = {"ERUPTION", "LABOR", "NEUROFIBROMATOSIS", "THYMUS", "Gamma-Interferon"}
EXCLUDED        = {"EGG", "STEM", "Orf"}

all_words    = sorted(w for w in concept_map if w not in EXCLUDED)
all_acronyms = [w for w in all_words if (is_acronym(w) or w in MANUAL_ACRONYMS) and w not in MANUAL_TERMS]
all_terms    = [w for w in all_words if w not in all_acronyms]

def load_instances(word):
    """Read all the sentences (instances) of one ambiguous word."""

    path = DATA_DIR / f"{word}_pmids_tagged.arff"
    lines = path.read_text(encoding="iso-8859-1").splitlines()

    # The sentences start right after the line "@DATA"
    start = next(i for i, l in enumerate(lines) if l.startswith("@DATA")) + 1

    instances = []
    for line in lines[start:]:
        line = line.strip()
        if not line:                       
            continue

        pmid, rest = line.split(",", 1)    # text before the FIRST comma = PMID
        text, label = rest.rsplit(",", 1)  # text after the LAST comma = label (M1, M2...)
        text = html.unescape(text.strip('"'))   # remove quotes, convert &lt; -> <

        # Remember which word was marked, then remove the <e> </e> tags
        marked = re.search(r"<e>(.+?)</e>", text)
        target = marked.group(1) if marked else word
        context = re.sub(r"</?e>", "", text)

        instances.append({
            "word": word,
            "pmid": pmid,
            "target": target,                          # e.g. "cements"
            "tagged": text,                            # e.g. "The <e>cements</e> were applied."
            "context": context,                        # clean sentence
            "label": label.strip(),                    # e.g. "M1"
            "cui": concept_map[word][label.strip()],   # correct sense, e.g. "C0002520"
        })
    return instances

Number of ambiguous words: 203
AA -> {'M1': 'C0002520', 'M2': 'C0001972'}
Ca -> {'M1': 'C0006675', 'M2': 'C0006754', 'M3': 'C0019564', 'M4': 'C0006823'}


In [44]:
# mapping of MSH-WSD senses (CUIs) to WordNet synsets, created by hand
MAPPING = {
    # ---------------- acronyms ----------------
    "ADP":   {"C0001459": ["adenosine_diphosphate.n.01"],      # Adenosine Diphosphate
              "C0004374": ["automatic_data_processing.n.01"]}, # Automatic Data Processing
    "CAD":   {"C0011905": [],                                  # Computer Assisted Diagnosis (WN has only "design")
              "C1956346": []},                                 # Coronary Artery Disease
    "CAM":   {"C0007578": [],                                  # Cell Adhesion Molecules
              "C0178551": []},                                 # chorioallantoic membrane
    "CDA":   {"C0092801": [],                                  # cladribine
              "C0002876": []},                                 # Congenital dyserythropoietic anemia
    "Ca":    {"C0006675": ["calcium.n.01"],                    # calcium
              "C0006754": ["california.n.01"],                 # California
              "C0019564": [],                                  # Hippocampus
              "C0006823": []},                                 # Canada
    "DDD":   {"C0026256": [],                                  # mitotane
              "C0011037": []},                                 # Dichlorodiphenyldichloroethane
    "DON":   {"C0028652": [],                                  # Nurse Administrator
              "C0012020": []},                                 # Diazooxonorleucine
    "SARS":  {"C1175175": ["severe_acute_respiratory_syndrome.n.01"],  # the disease
              "C1175743": []},                                 # the coronavirus
    "Staph": {"C0038160": [],                                  # Staphylococcal Infections
              "C0038170": ["staphylococcus.n.01"]},            # Genus staphylococcus
    "TNT":   {"C0077404": [],                                  # Troponin T
              "C0041070": ["tnt.n.01"]},                       # Trinitrotoluene
    "ALS": {"C0003372": [],                                         # Antilymphocyte Serum
            "C0002736": ["amyotrophic_lateral_sclerosis.n.01"]},    # Amyotrophic Lateral Sclerosis
    "ANA": {"C0002463": [],                                         # American Nurses' Association
            "C0003243": []},                                        # Antinuclear Antibodies
    "CI":  {"C0022326": [],                                         # Ivory Coast
            "C0008107": []},                                        # Chile
    "CLS": {"C0265252": [],                                         # Coffin-Lowry syndrome
            "C0343084": []},                                        # Capillary Leak Syndrome
    "FAS": {"C0015683": [],                                         # Fatty-acid synthase
            "C0015923": ["fetal_alcohol_syndrome.n.01"]},           # Fetal Alcohol Syndrome
    "GAG": {"C0017346": [],                                         # GAG Gene
            "C0017973": []},                                        # Glycosaminoglycans
    "MRS": {"C0025235": [],                                         # Melkersson-Rosenthal Syndrome
            "C0024487": []},                                        # Magnetic Resonance Spectroscopy
    "PVC": {"C0151636": ["premature_ventricular_contraction.n.01"], # Premature ventricular contractions
            "C0032624": ["polyvinyl_chloride.n.01"]},               # Polyvinyl Chloride
    "RA":  {"C0002893": [],                                         # Refractory anemias
            "C0034625": ["radium.n.01"],                            # radium
            "C0003873": []},                                        # Rheumatoid Arthritis
    "US":  {"C0041703": ["united_states.n.01"],                     # United States
            "C0041618": []},                                        # Ultrasonography
    # ---------------- regular terms ----------------
    "B-Cell Leukemia": {"C2004493": [],                        # Leukemia, B-Cell
                        "C0023434": []},                       # Chronic Lymphocytic Leukemia
    "Borrelia":    {"C0024198": [],                            # Lyme Disease
                    "C0006033": ["borrelia.n.01"]},            # Borrelia (bacterium)
    "Cilia":       {"C0008778": ["cilium.n.01"],               # Cilia (cell projection)
                    "C0015422": ["eyelash.n.01"]},             # Eyelash
    "Follicles":   {"C0221971": [],                            # Hair follicle
                    "C0018120": ["follicle.n.01"]},            # Ovarian Follicle (closest to WN gloss)
    "Ganglion":    {"C0017067": ["ganglion.n.01"],             # Ganglia (neural)
                    "C1258666": []},                           # Myxoid cyst
    "Nurse":       {"C0028661": ["nurse.n.01", "nurse.v.03"],  # Nurses (the profession)
                    "C0006147": ["breastfeed.v.01"]},          # Breast Feeding
    "Pneumocystis": {"C0032305": [],                           # (retired CUI, name not available)
                     "C0597258": []},                          # Genus Pneumocystis
    "Respiration": {"C0035203": ["respiration.n.02", "breathing.n.01"],  # breathing
                    "C0282636": ["respiration.n.01"]},         # Cell Respiration (metabolic)
    "Sodium":      {"C0037473": ["sodium.n.01"],               # sodium (element)
                    "C0037570": []},                           # Sodium, Dietary
    "Synapsis":    {"C0039062": [],                            # Synapses (neuron junction)
                    "C0598501": ["synapsis.n.01"]},            # Chromosome Pairing
    "Callus":     {"C0006767": ["callus.n.02"],                     # Bone callus
                   "C0376154": ["callosity.n.01"]},                 # Skin callus
    "Cell":       {"C0007634": ["cell.n.02"],                       # Cells (biology)
                   "C1136359": ["cellular_telephone.n.01"]},        # Cellular Phone
    "Cement":     {"C1706094": ["cement.n.04", "cement.n.03"],      # Dental Cement Material
                   "C0011343": ["cementum.n.01"]},                  # Dental Cementum
    "ERUPTION":   {"C0015230": ["eruption.n.02"],                   # Exanthema (skin rash)
                   "C1533692": ["eruption.n.06"]},                  # Tooth eruption
    "Exercises":  {"C0452240": ["exercise.n.03"],                   # Physical therapy exercises
                   "C0015259": ["exercise.n.01"]},                  # Exercise
    "Medullary":  {"C0001629": ["medullary.a.03"],                  # Adrenal Medulla
                   "C0025148": ["medullary.a.02"]},                 # Medulla Oblongata
    "Milk":       {"C0026131": ["milk.n.01"],                       # Milk (body substance)
                   "C0026140": ["milk.n.02"]},                      # Milk, Human
    "Phosphorus": {"C0080014": [],                                  # Dietary Phosphorus
                   "C0031705": ["phosphorus.n.01"]},                # phosphorus (element)
    "Ventricles": {"C0018827": ["ventricle.n.02"],                  # Heart Ventricle
                   "C0007799": ["ventricle.n.01"]},                 # Cerebral Ventricles
}

ACRONYMS = [w for w in all_acronyms if w in MAPPING]
TERMS    = [w for w in all_terms if w in MAPPING]

print(len(ACRONYMS), "acronyms:", ACRONYMS)
print(len(TERMS), "terms:", TERMS)

20 acronyms: ['ADP', 'ALS', 'ANA', 'CAD', 'CAM', 'CDA', 'CI', 'CLS', 'Ca', 'DDD', 'DON', 'FAS', 'GAG', 'MRS', 'PVC', 'RA', 'SARS', 'Staph', 'TNT', 'US']
19 terms: ['B-Cell Leukemia', 'Borrelia', 'Callus', 'Cell', 'Cement', 'Cilia', 'ERUPTION', 'Exercises', 'Follicles', 'Ganglion', 'Medullary', 'Milk', 'Nurse', 'Phosphorus', 'Pneumocystis', 'Respiration', 'Sodium', 'Synapsis', 'Ventricles']


In [45]:
def wordnet_senses(word):
    """NLTK WordNet senses of an MSH-WSD word, used only by NLTK Lesk"""
    w = word.lower().replace(" ", "_")
    senses = wn.synsets(w)
    if not senses:
        base = wn.morphy(w)
        if base:
            senses = wn.synsets(base)
    return senses

get_key = sensekey.sense_key_getter("oewn:2024")   # function: pywsd sense -> sense key

def to_nltk(pywsd_synset):
    """Translate a pywsd synset (e.g. oewn-01460746-n) into its NLTK name (e.g. cilium.n.01), through the sense key."""
    if pywsd_synset is None:
        return None
    for sense in pywsd_synset.senses():
        key = get_key(sense)
        if key:
            try:
                return wn.lemma_from_key(key).synset().name()
            except Exception:
                continue
    return None

def pywsd_form(word):                                                  
    """Spelling of the word that lets pywsd see most of the senses used in MAPPING."""
    mapped = {s for synsets in MAPPING[word].values() for s in synsets}  
    best_form, best_score = word, (-1, -1)
    for form in dict.fromkeys([word, word.lower(), word.capitalize(), word.upper()]):
        found = {to_nltk(s) for s in pwn.synsets(lemmatize(form))}
        score = (len(found & mapped), len(found))
        if score > best_score:
            best_form, best_score = form, score
    return best_form


for word in ACRONYMS + TERMS:
    print(f"{word:16} -> {pywsd_form(word)!r}")

ADP              -> 'ADP'
ALS              -> 'ALS'
ANA              -> 'ANA'
CAD              -> 'cad'
CAM              -> 'CAM'
CDA              -> 'CDA'
CI               -> 'CI'
CLS              -> 'CLS'
Ca               -> 'ca'
DDD              -> 'DDD'
DON              -> 'DON'
FAS              -> 'FAS'
GAG              -> 'GAG'
MRS              -> 'MRS'
PVC              -> 'PVC'
RA               -> 'ra'
SARS             -> 'SARS'
Staph            -> 'Staph'
TNT              -> 'TNT'
US               -> 'US'
B-Cell Leukemia  -> 'B-Cell Leukemia'
Borrelia         -> 'Borrelia'
Callus           -> 'Callus'
Cell             -> 'Cell'
Cement           -> 'Cement'
Cilia            -> 'Cilia'
ERUPTION         -> 'ERUPTION'
Exercises        -> 'exercises'
Follicles        -> 'follicles'
Ganglion         -> 'Ganglion'
Medullary        -> 'Medullary'
Milk             -> 'milk'
Nurse            -> 'Nurse'
Phosphorus       -> 'phosphorus'
Pneumocystis     -> 'Pneumocystis'
Respiration      -

In [46]:
WINDOW = 10    

def context_window(tagged_text, n=WINDOW):
    """Return the n words before the target, the target itself and the n words after it."""
    if "<e>" not in tagged_text:                
        return re.sub(r"</?e>", "", tagged_text)

    before, rest = tagged_text.split("<e>", 1)      
    target, after = rest.split("</e>", 1)             

    words_before = re.findall(r"\w+", before)[-n:]    
    words_after = re.findall(r"\w+", re.sub(r"</?e>", "", after))[:n]   

    return " ".join(words_before + [target] + words_after)            


# Check on one abstract
example = load_instances("AA")[0]
print("Full abstract :", len(example["context"].split()), "words")
print("Window        :", context_window(example["tagged"]))

Full abstract : 262 words
Window        : effect of the duration of prefeeding on prececal amino acid AA digestibility of wheat distillers dried grains with solubles DDGS in


In [47]:
N_PER_SENSE = 10   # abstracts sampled for each sense of each word

def sample_instances(word, n=N_PER_SENSE, seed=0):
    """Random sample of n abstracts for each sense (CUI) of the word."""
    rng = random.Random(seed)
    by_sense = {}
    for inst in load_instances(word):
        by_sense.setdefault(inst["cui"], []).append(inst)
    sample = []
    for cui in sorted(by_sense):
        sample += rng.sample(by_sense[cui], min(n, len(by_sense[cui])))
    return sample


n_abstracts = sum(len(sample_instances(w)) for w in ACRONYMS + TERMS)
print("Abstracts in the sample:", n_abstracts)

Abstracts in the sample: 810


In [48]:
def accuracy(rows, word, method, covered_only=False):
    if covered_only:
        rows = [r for r in rows if MAPPING[word][r["cui"]]]
    if not rows:
        return None
    correct = [r["answers"][method] in MAPPING[word][r["cui"]] for r in rows]
    return sum(correct) / len(correct)


def mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else float("nan")

In [49]:
ALPHAS = [1, 0.8, 0.5]


def wsd_alphas(context, word):
    """Best sense for each alpha. Wu-Palmer and phonetic values are computed only once and reused."""
    ctx = [lemmatize(w) for w in word_tokenize(context)]
    target = lemmatize(word)
    ctx = [w for w in ctx if w.lower() != target.lower()]
    totals = {a: [] for a in ALPHAS}
    for i in pwn.synsets(target):
        pairs = [[(sim(i, k, "wup") or 0.0, sense_phonetic(i, k)) for k in pwn.synsets(j)] for j in ctx]
        for a in ALPHAS:
            total = sum(max([a * w + (1 - a) * p for w, p in pairs_j], default=0.0) for pairs_j in pairs)
            totals[a].append((total, i))
    return {str(a): to_nltk(max(t, key=lambda x: x[0])[1]) if t else None for a, t in totals.items()}

In [50]:
CACHE6 = Path("task6_predictions.json")
predictions = json.loads(CACHE6.read_text()) if CACHE6.exists() else {}

for word in ACRONYMS + TERMS:
    if word in predictions:
        continue
    t = time.time()
    covered = any(MAPPING[word].values())
    rows = []
    for inst in sample_instances(word):
        if covered:
            answers = wsd_alphas(context_window(inst["tagged"]), pywsd_form(word))
        else:
            answers = {str(a): None for a in ALPHAS}
        rows.append({"cui": inst["cui"], "answers": answers})
    predictions[word] = rows
    CACHE6.write_text(json.dumps(predictions))
    print(f"{word:16} {len(rows):3} abstracts {time.time() - t:6.1f} s")

print("Done.")

Done.


In [51]:
pred4 = json.loads(Path("../p3&4/task4_predictions.json").read_text())

compare = [("Wu-Palmer (pywsd, Task 4)", pred4, "Wu-Palmer")]
compare += [(f"alpha = {a}", predictions, str(a)) for a in ALPHAS]

table = []
for name, preds, method in compare:
    line = {"method": name}
    for group_name, group in [("acronyms", ACRONYMS), ("terms", TERMS)]:
        line[f"{group_name} acc"] = mean(accuracy(preds[w], w, method) for w in group)
        line[f"{group_name} cov"] = mean(accuracy(preds[w], w, method, covered_only=True) for w in group)
    table.append(line)

results6 = pd.DataFrame(table).set_index("method")
results6.round(3)

,acronyms acc,acronyms cov,terms acc,terms cov
method,,,,
"Wu-Palmer (pywsd, Task 4)",0.250,0.905,0.345,0.562
alpha = 1,0.248,0.895,0.345,0.562
alpha = 0.8,0.249,0.895,0.332,0.547
alpha = 0.5,0.265,0.920,0.342,0.559


**Observations:**
- `alpha = 0.5` is a bit better on acronyms (0.248 → 0.265), but on terms the best is still `alpha = 1` (0.345).
- The differences are very small and `alpha = 0.8` is not between the other two, so they depend on a few abstracts and are not significant.
- So, on average, adding the phonetic similarity does not really improve Wu-Palmer.

The averages hide what happens on each word. Here we print only the words where the alpha changes the accuracy, to see where the phonetic similarity helps and where it makes things worse.

In [52]:
for w in ACRONYMS + TERMS:
    accs = {a: accuracy(predictions[w], w, str(a)) for a in ALPHAS}
    if len(set(accs.values())) > 1:          # only the words where alpha changes the result
        print(f"{w:12} " + "  ".join(f"alpha={a}: {accs[a]:.2f}" for a in ALPHAS))

ADP          alpha=1: 0.70  alpha=0.8: 0.65  alpha=0.5: 0.85
Ca           alpha=1: 0.33  alpha=0.8: 0.30  alpha=0.5: 0.30
PVC          alpha=1: 0.60  alpha=0.8: 0.70  alpha=0.5: 0.85
RA           alpha=1: 0.33  alpha=0.8: 0.33  alpha=0.5: 0.30
Cell         alpha=1: 0.20  alpha=0.8: 0.10  alpha=0.5: 0.25
Cement       alpha=1: 0.40  alpha=0.8: 0.45  alpha=0.5: 0.45
Cilia        alpha=1: 0.20  alpha=0.8: 0.45  alpha=0.5: 0.50
ERUPTION     alpha=1: 0.25  alpha=0.8: 0.00  alpha=0.5: 0.00
Exercises    alpha=1: 0.00  alpha=0.8: 0.05  alpha=0.5: 0.10
Milk         alpha=1: 0.60  alpha=0.8: 0.60  alpha=0.5: 0.55
Nurse        alpha=1: 0.20  alpha=0.8: 0.05  alpha=0.5: 0.15
Respiration  alpha=1: 0.80  alpha=0.8: 0.70  alpha=0.5: 0.65
Ventricles   alpha=1: 0.50  alpha=0.8: 0.50  alpha=0.5: 0.45


**Observations:**
- The phonetic part changes the result only on 13 of the 39 words, and it helps on some words and makes things worse on others.
- It helps a lot on two acronyms, **ADP** (0.70 → 0.85) and **PVC** (0.60 → 0.85), and this is the reason why acronyms improve on average. A Soundex code has only 4 characters, so the code of a long name is the same as the code of its first word: *adenosine diphosphate* = *adenosine* (A352), *premature ventricular contraction* = *premature* (P653). When the abstract contains *adenosine* or *premature*, the correct sense gets a phonetic similarity of 1.
- It makes things worse on terms like **ERUPTION** (0.25 → 0.00) and **Respiration** (0.80 → 0.65): here all the senses have the same name, so the phonetic part cannot separate them and only adds noise, as with *dose* in the *drug* example.
- **Conclusion:** the phonetic similarity is not related to meaning. It helps only when the name of the correct sense contains a word of the context, so it is not a good general improvement for WSD.